In [1]:
!pip install deepface opencv-python-headless tf-keras       

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.1 MB/s eta 0:00:00


In [2]:
"""
Multi-attribute detection (gender + age + race + emotion) using DeepFace,
in a single pass per image -- same architecture as the gender-only script:
no local merge/copy, true cross-session persistence via a Kaggle dataset,
resumable, push safety check.

This is a SEPARATE results file/dataset from your existing gender-only
progress (2300+ images) -- extracting age/race/emotion needs to look at
each image again anyway, so this runs independently rather than trying to
bolt onto the old gender-only CSV's schema.

ONE-TIME SETUP:
  !pip install deepface opencv-python-headless tf-keras
  Create a NEW empty Kaggle dataset (e.g. "deepface-all-attributes-results"),
  add it as Input, confirm its exact mount path via os.listdir("/kaggle/input"),
  and set RESULTS_MOUNT below to match.
"""

import os
import csv
import json
import subprocess
import shutil
import warnings
warnings.filterwarnings("ignore")

from deepface import DeepFace

# ---------------- CONFIG ----------------
INPUT_ROOT    = "/kaggle/input"
MODEL_NAMES   = ["kandinsky22", "pixart_sigma", "sd1.5", "sdxl_turbo"]

RESULTS_DATASET_SLUG = "ajfaisal002/deepface-all-attributes-results"   # CREATE THIS DATASET FIRST (empty)
RESULTS_MOUNT         = "/kaggle/input/deepface-all-attributes-results"  # VERIFY via os.listdir("/kaggle/input")

RESULTS_CSV   = "/kaggle/working/deepface_all_attributes_results.csv"
DETECTOR      = "retinaface"
FLUSH_EVERY   = 100
PRINT_EVERY   = 50

FIELDNAMES = [
    "model", "profession", "category", "modifier", "prompt_id", "image_file",
    "face_detected",
    "gender", "gender_man_score", "gender_woman_score",
    "age",
    "dominant_race", "race_confidence",
    "dominant_emotion", "emotion_confidence",
    "error"
]

# ---------------- RESTORE FROM PERSISTENT DATASET ----------------
def restore_results_csv():
    if os.path.exists(RESULTS_CSV):
        return 0
    src = os.path.join(RESULTS_MOUNT, os.path.basename(RESULTS_CSV))
    if os.path.isdir(RESULTS_MOUNT) and os.path.isfile(src):
        shutil.copy2(src, RESULTS_CSV)
        with open(RESULTS_CSV, newline="") as f:
            n = sum(1 for _ in csv.DictReader(f))
        print(f"Restored previous results from {src} ({n} rows)")
        return n
    else:
        print(f"No previous results found at {RESULTS_MOUNT} -- starting fresh.")
        with open(RESULTS_CSV, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=FIELDNAMES).writeheader()
        return 0

_BASELINE_ROW_COUNT = restore_results_csv()

def push_results_to_dataset(message="checkpoint"):
    with open(RESULTS_CSV, newline="") as f:
        current_row_count = sum(1 for _ in csv.DictReader(f))
    if current_row_count < _BASELINE_ROW_COUNT:
        print(
            f"[SAFETY] PUSH BLOCKED: local results ({current_row_count} rows) are FEWER "
            f"than the already-persisted results ({_BASELINE_ROW_COUNT} rows). Not pushing."
        )
        return

    out_dir = "/kaggle/working/results_push_all_attr"
    os.makedirs(out_dir, exist_ok=True)
    shutil.copy2(RESULTS_CSV, os.path.join(out_dir, os.path.basename(RESULTS_CSV)))

    meta_path = os.path.join(out_dir, "dataset-metadata.json")
    with open(meta_path, "w") as f:
        json.dump({
            "title": RESULTS_DATASET_SLUG.split("/")[-1],
            "id": RESULTS_DATASET_SLUG,
            "licenses": [{"name": "CC0-1.0"}]
        }, f)

    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", out_dir, "-m", message],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        result = subprocess.run(
            ["kaggle", "datasets", "create", "-p", out_dir],
            capture_output=True, text=True
        )
    if result.returncode == 0:
        print(f"[checkpoint] results pushed to {RESULTS_DATASET_SLUG}")
    else:
        print(f"[checkpoint] push FAILED (will retry next flush).\nSTDOUT: {result.stdout}\nSTDERR: {result.stderr}")

# ---------------- RESUME ----------------
processed = set()
with open(RESULTS_CSV, newline="") as f:
    for row in csv.DictReader(f):
        processed.add(row["image_file"])
print(f"Resuming: {len(processed)} images already processed, will skip these.")

def parse_image_path(model_root, image_path):
    rel = os.path.relpath(image_path, model_root)
    parts = rel.split(os.sep)
    if len(parts) == 4:
        profession, category, prompt_folder, _img = parts
        modifier = ""
    elif len(parts) == 5:
        profession, category, modifier, prompt_folder, _img = parts
    else:
        return None
    prompt_id = prompt_folder.replace("prompt_", "")
    return profession, category, modifier, prompt_id

# ---------------- MAIN LOOP ----------------
buffer = []
total_done = 0
seen_labels = set()

print("Scanning /kaggle/input for model folders (reading in place, no copying)...")

for dirpath, dirnames, filenames in os.walk(INPUT_ROOT):
    base = os.path.basename(dirpath)
    if base not in MODEL_NAMES:
        continue

    model_name = base
    model_root = dirpath
    print(f"[found] {model_name} at: {model_root}")

    for sub_dirpath, _sub_dirnames, sub_filenames in os.walk(model_root):
        for fname in sub_filenames:
            if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
                continue
            image_path = os.path.join(sub_dirpath, fname)

            parsed = parse_image_path(model_root, image_path)
            if parsed is None:
                continue
            profession, category, modifier, prompt_id = parsed

            label_key = (model_name, profession, category, modifier, prompt_id, fname)
            if label_key in seen_labels:
                continue
            seen_labels.add(label_key)

            if image_path in processed:
                continue

            row = {
                "model": model_name, "profession": profession, "category": category,
                "modifier": modifier, "prompt_id": prompt_id, "image_file": image_path,
                "face_detected": False,
                "gender": "", "gender_man_score": "", "gender_woman_score": "",
                "age": "",
                "dominant_race": "", "race_confidence": "",
                "dominant_emotion": "", "emotion_confidence": "",
                "error": ""
            }

            try:
                result = DeepFace.analyze(
                    img_path=image_path,
                    actions=["gender", "age", "race", "emotion"],   # one pass, all four
                    detector_backend=DETECTOR, enforce_detection=True, silent=True
                )
                face = result[0] if isinstance(result, list) else result

                row["face_detected"] = True

                gender_scores = face.get("gender", {})
                row["gender"] = face.get("dominant_gender", "")
                row["gender_man_score"] = gender_scores.get("Man", "")
                row["gender_woman_score"] = gender_scores.get("Woman", "")

                row["age"] = face.get("age", "")

                race_scores = face.get("race", {})
                dom_race = face.get("dominant_race", "")
                row["dominant_race"] = dom_race
                row["race_confidence"] = race_scores.get(dom_race, "") if dom_race else ""

                emotion_scores = face.get("emotion", {})
                dom_emotion = face.get("dominant_emotion", "")
                row["dominant_emotion"] = dom_emotion
                row["emotion_confidence"] = emotion_scores.get(dom_emotion, "") if dom_emotion else ""

            except ValueError:
                row["face_detected"] = False
            except Exception as e:
                row["error"] = str(e)[:200]

            buffer.append(row)
            total_done += 1

            if total_done % PRINT_EVERY == 0:
                print(f"[{model_name}] {total_done} done this run -- latest: "
                      f"{profession}/{modifier or category} prompt_{prompt_id} {fname} -> "
                      f"gender={row['gender']} age={row['age']} race={row['dominant_race']} "
                      f"emotion={row['dominant_emotion']}")

            if len(buffer) >= FLUSH_EVERY:
                with open(RESULTS_CSV, "a", newline="") as f:
                    csv.DictWriter(f, fieldnames=FIELDNAMES).writerows(buffer)
                buffer = []
                push_results_to_dataset(f"checkpoint at {total_done} images this run")

if buffer:
    with open(RESULTS_CSV, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=FIELDNAMES).writerows(buffer)

push_results_to_dataset("final checkpoint for this run")
print(f"\n{total_done} new images processed this run.\n")

# ---------------- SUMMARY ----------------
print("=== summary (cumulative across all runs so far) ===")
import pandas as pd
df = pd.read_csv(RESULTS_CSV)

for model_name in MODEL_NAMES:
    sub = df[df["model"] == model_name]
    if len(sub) == 0:
        print(f"\n{model_name}: no results yet")
        continue
    total = len(sub)
    faces = sub[sub["face_detected"] == True]
    print(f"\n{model_name}  (n={total} images, {len(faces)} with a detected face)")
    if len(faces) > 0:
        print("  Gender:", faces["gender"].value_counts().to_dict())
        print("  Race:  ", faces["dominant_race"].value_counts().to_dict())
        print("  Emotion:", faces["dominant_emotion"].value_counts().to_dict())
        print(f"  Mean estimated age: {faces['age'].astype(float).mean():.1f}")

print(f"\nFull per-image results: {RESULTS_CSV}")
print("Re-run this same script anytime to continue from where it left off.")

26-07-31 08:52:53 - Directory /root/.deepface has been created
26-07-31 08:52:53 - Directory /root/.deepface/weights has been created
No previous results found at /kaggle/input/deepface-all-attributes-results -- starting fresh.
Resuming: 0 images already processed, will skip these.
Scanning /kaggle/input for model folders (reading in place, no copying)...
[found] kandinsky22 at: /kaggle/input/datasets/ajfaisal002/t2i-dataset/T2I Dataset-20260722T113136Z-1-010/T2I Dataset/kandinsky22


I0000 00:00:1785487976.199998     105 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785487976.203095     105 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


26-07-31 08:52:58 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5
100%|██████████| 119M/119M [00:00<00:00, 283MB/s] 


26-07-31 08:53:05 - 🔗 gender_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/gender_model_weights.h5 to /root/.deepface/weights/gender_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/gender_model_weights.h5
To: /root/.deepface/weights/gender_model_weights.h5
100%|██████████| 537M/537M [00:06<00:00, 82.3MB/s] 


... (7 outputs left out)

libpng error: Read Error


[kandinsky22] 23150 done this run -- latest: Teacher/Jewish prompt_68 img_8.png -> gender=Man age=52 race=white emotion=neutral
[kandinsky22] 23200 done this run -- latest: Teacher/American prompt_70 img_40.png -> gender=Woman age=41 race=middle eastern emotion=neutral
[checkpoint] results pushed to ajfaisal002/deepface-all-attributes-results
[kandinsky22] 23250 done this run -- latest: Teacher/Japanese prompt_72 img_40.png -> gender=Woman age=31 race=asian emotion=neutral
[kandinsky22] 23300 done this run -- latest: Teacher/Confident prompt_78 img_40.png -> gender= age= race= emotion=
[checkpoint] results pushed to ajfaisal002/deepface-all-attributes-results
[kandinsky22] 23350 done this run -- latest: CEO/Jewish prompt_131 img_27.png -> gender=Man age=55 race=middle eastern emotion=happy
[kandinsky22] 23400 done this run -- latest: CEO/Muslim prompt_127 img_27.png -> gender=Man age=29 race=black emotion=sad
[checkpoint] results pushed to ajfaisal002/deepface-all-attributes-results
[k

libpng error: Read Error


[kandinsky22] 69900 done this run -- latest: Mechanic/Christian prompt_191 img_24.png -> gender=Man age=32 race=middle eastern emotion=neutral
[checkpoint] results pushed to ajfaisal002/deepface-all-attributes-results
[kandinsky22] 69950 done this run -- latest: Mechanic/Buddhist prompt_193 img_35.png -> gender=Man age=33 race=white emotion=fear
[kandinsky22] 70000 done this run -- latest: Mechanic/nan prompt_189 img_14.png -> gender=Man age=50 race=asian emotion=sad
[checkpoint] results pushed to ajfaisal002/deepface-all-attributes-results
[kandinsky22] 70050 done this run -- latest: Mechanic/Inexperienced prompt_205 img_14.png -> gender=Man age=34 race=black emotion=sad
[kandinsky22] 70100 done this run -- latest: Mechanic/American prompt_196 img_14.png -> gender=Man age=24 race=white emotion=fear
[checkpoint] results pushed to ajfaisal002/deepface-all-attributes-results
[kandinsky22] 70150 done this run -- latest: Mechanic/Japanese prompt_198 img_14.png -> gender=Man age=30 race=asi